# Hybrid Quantum–Classical Methods — Detailed Notes (Session 13)
**Course:** CS490/5590 — Quantum Computing Applications in Data Science, AI, & Deep Learning  
**Instructor:** Luke Miller  

> **Purpose.** These notes turn the slide bullets into a stand-alone reference on building end-to-end **hybrid** ML pipelines that combine small quantum circuits with classical models and optimizers. You’ll learn interface patterns (encode → PQC → decode), training loops, and modern Qiskit + PyTorch/TensorFlow integrations, with code templates and pitfalls to avoid.

---

## Session road-map
1. Recap: QNNs and why hybridize  
2. Interface design: encode → **quantum** → decode → **classical**  
3. Data **encoding** (amplitude, angle, entangling maps)  
4. Data **decoding** (expectations, probabilities, kernels)  
5. Hybrid optimization loops (parameter-shift, SPSA, batching)  
6. Deep-learning use cases (feature heads, hybrid stacks, subroutines)  
7. Qiskit ML + **PyTorch** (official connector) & a minimal **TensorFlow** pattern  
8. Practical guidance: noise, latency, caching, calibration  
9. Mini-exercises & lab outline

---

## 0) Why hybrid?
- **Today’s constraint:** few qubits + noise → quantum alone won’t scale.  
- **Hybrid idea:** let the quantum part do *what it is uniquely good at* (compact, entangling feature maps; non-classical similarity), and let the classical part do the rest (large-scale optimization, representation learning, data I/O).  
- **Win:** shorter circuits, smaller shot budgets, better gradient signal-to-noise, and simpler deployment.

**Common pattern**  
$$
x\in\mathbb{R}^d \xrightarrow{\text{encode}} U_\phi(x)\,|0\rangle^{\otimes n}
\xrightarrow{\text{PQC }U(\theta)} |\psi\rangle
\xrightarrow{\text{measure}} z \in \mathbb{R}^m
\xrightarrow{\text{classical}} \hat{y}
$$

---

## 1) Quantum–classical interfaces

### Quantum → Classical (decoding)
- **Expectations:** $z_i=\langle P_i\rangle$ (e.g., $P_i=Z_j$) → logits/regression targets.
- **Probabilities:** histogram over bitstrings → soft decisions / uncertainty.
- **Kernels:** $K(x,x')=|\langle\phi(x)|\phi(x')\rangle|^2$ → classical SVM.

### Classical → Quantum (encoding & control)
- **Data encoding:** map $x$ into gate angles or amplitudes.  
- **Parameter updates:** classical optimizer proposes $\theta$ (and optionally re-uploads $x$).

**Bottlenecks:** measurement shot noise, device latency, and transpilation overhead.  
**Mitigations:** readout calibration, batching, result caching, lower-depth ansätze.

---

## 2) Data encoding (choose by budget)

| Encoding | Qubits | Depth | Pros | Cons | Good for |
|---|---:|---:|---|---|---|
| **Angle** ($R_Y(\alpha x_i)$, $R_Z(\beta x_i)$) | $\approx d$ | low | Simple, hardware-friendly | 1 qubit / feature | tabular, 2–8 features |
| **Entangling angle** (ZZFeatureMap) | $\approx d$ | low–mid | Non-linear interactions | extra 2-qubit gates | small graphs/images |
| **Amplitude** ($|x\rangle/\|x\|$) | $\lceil\log_2 d\rceil$ | mid–high | Exponentially compact | Costly state-prep | very high-dim data |
| **Data re-uploading** | fixed | mid | More expressivity | more evals/grad cost | few-qubit hardware |

> **Tip:** Standardize/scale inputs so angles lie in $[-\pi,\pi]$. With more than ~8 raw features, **select features first** (Session 11).

---

## 3) Data decoding (what to measure)

- **Classifier (binary):** $ \hat{p} = \frac{1-\langle Z_0\rangle}{2} $ → BCE or hinge loss.  
- **Regressor:** $ f(x) = \sum_i w_i \langle Z_i\rangle $ → MSE.  
- **Multiclass:** multiple readout qubits → vector of expectations → softmax/classical head.  
- **Uncertainty:** use sample variance of measured expectation or full probability vector.

**Calibration:** apply confusion-matrix inversion for readout; normalise outputs before feeding dense layers.

---

## 4) Hybrid optimization loops

### Parameter-shift gradient (exact for many rotations)
$$
\frac{\partial \langle H\rangle}{\partial \theta_k}
=\tfrac12\Big(\langle H\rangle_{\theta_k+\pi/2}-\langle H\rangle_{\theta_k-\pi/2}\Big)
$$
- Cost: **2 evaluations per parameter** (× shots).  
- Works well on simulators and low-noise hardware.

### Gradient-free options
- **SPSA:** 2 evaluations per step regardless of dimension; robust to shot noise.  
- **COBYLA/Nelder–Mead:** ok for small parameter counts.

### Batching & caching
- Batch inputs for a given $\theta$ to amortize latency.  
- Cache **transpiled** circuits; only bind parameters (fast path).

---

## 5) Hybrid use-cases in deep learning

1. **Quantum feature head + classical body**  
   - PQC produces 2–8 features → classical MLP/CNN finishes.  
2. **Classical feature extractor + quantum head**  
   - Small CNN or PCA compresses to 4–6 dims → encode to PQC → decision.  
3. **Quantum subroutines inside training**  
   - QAOA/VQE for inner optimizations (e.g., discrete layer selection).  
4. **Kernel pipelines**  
   - Quantum kernel → classical SVM; hybrid selection reduces qubits/kernels.

---

## 6) Qiskit ML + **PyTorch** (recommended path)

> Modern Qiskit ML uses **primitives** (`Estimator`, `Sampler`) with **EstimatorQNN**/**SamplerQNN**.  
> The legacy `CircuitQNN`/Opflow APIs are deprecated.

### 6.1 EstimatorQNN + TorchConnector (binary classification)
```python
# pip install qiskit qiskit-aer qiskit-machine-learning torch scikit-learn
from qiskit.circuit.library import ZZFeatureMap, TwoLocal
from qiskit import QuantumCircuit
from qiskit_aer.primitives import Estimator
from qiskit_machine_learning.neural_networks import EstimatorQNN
from qiskit_machine_learning.connectors import TorchConnector

import torch
from sklearn.datasets import make_moons
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

# 1) data
X, y = make_moons(n_samples=200, noise=0.15, random_state=0)
X = StandardScaler().fit_transform(X)
Xtr, Xte, ytr, yte = train_test_split(X, y, test_size=0.3, random_state=0)
Xtr = torch.tensor(Xtr, dtype=torch.float32); ytr = torch.tensor(ytr, dtype=torch.float32)
Xte  = torch.tensor(Xte,  dtype=torch.float32); yte  = torch.tensor(yte,  dtype=torch.float32)

# 2) quantum model (2 qubits)
feature_map = ZZFeatureMap(feature_dimension=2, reps=1)
ansatz      = TwoLocal(2, rotation_blocks=['ry','rz'], entanglement_blocks='cx', reps=2)
qc = QuantumCircuit(2)
qc = qc.compose(feature_map).compose(ansatz)

# 3) QNN + torch bridge
qnn = EstimatorQNN(circuit=qc,
                   input_params=feature_map.parameters,
                   weight_params=ansatz.parameters,
                   estimator=Estimator(shots=2048))
model = TorchConnector(qnn)                 # torch.nn.Module

# 4) wrap with a small torch head for BCE
clf = torch.nn.Sequential(model, torch.nn.Sigmoid())
optim = torch.optim.Adam(clf.parameters(), lr=0.05)
lossf = torch.nn.BCELoss()

for epoch in range(60):
    optim.zero_grad()
    pred = clf(Xtr).squeeze()
    loss = lossf(pred, ytr)
    loss.backward()
    optim.step()

with torch.no_grad():
    acc = ((clf(Xte).squeeze() > 0.5) == yte.bool()).float().mean().item()
print("Test accuracy:", round(acc, 3))
```

**Notes**
- `Estimator(shots=...)` controls stochasticity; increase shots when gradients stabilize.  
- To run on hardware, swap the primitive’s backend.  
- For multiclass, output a vector of expectations and apply a classical linear/softmax head.

---

## 7) Minimal **TensorFlow/Keras** pattern

Qiskit ML doesn’t ship an official Keras connector. The simple approach is to wrap the QNN as a `tf.keras.layers.Layer` and call it via `tf.numpy_function` (non-differentiable) **or** train QNN weights in PyTorch and use TF for other layers. For fully TF-native autodiff, use PennyLane or write a custom gradient; here’s a pragmatic wrapper for inference or two-phase training:

```python
import tensorflow as tf
import numpy as np
from qiskit_aer.primitives import Estimator
from qiskit_machine_learning.neural_networks import EstimatorQNN
# ... build the same `qnn` as above ...

class QNNLayer(tf.keras.layers.Layer):
    def __init__(self, qnn: EstimatorQNN):
        super().__init__(); self.qnn = qnn
        # mirror quantum weight params as Keras variables
        self.theta = tf.Variable(np.random.uniform(-0.1,0.1,len(qnn.weight_params)),
                                 dtype=tf.float32, trainable=True)

    def call(self, x):
        # x shape: [batch, feature_dim]; returns [batch, 1]
        def _eval(batch_np, theta_np):
            # batch of inputs -> list of expectation values
            outs = [self.qnn.forward(input_data=inp, weights=theta_np).item()
                    for inp in batch_np]
            return np.array(outs, dtype=np.float32)[:, None]
        y = tf.numpy_function(_eval, [x, self.theta], tf.float32)
        y.set_shape([None, 1])
        return y
```

> This wrapper **won’t propagate gradients** through the quantum layer automatically; you’d train `theta` with a separate optimizer loop (parameter-shift/SPSA) and let Keras train the surrounding classical layers. If you need end-to-end TF autodiff, consider alternatives (e.g., PennyLane’s Keras layers) or a custom gradient with `@tf.custom_gradient`.

---

## 8) Practical guidance

- **Latency dominates**: batch inputs per parameter vector; reuse transpiled circuits (bind parameters only).  
- **Calibrate readout**: reduces bias in $\langle Z\rangle$ and downstream BCE/MSE.  
- **Mitigate noise**: dynamical decoupling on idle qubits; ZNE for critical evaluations (few checkpoints).  
- **Normalise outputs**: standardise expectation vectors before classical dense layers.  
- **Tune shots**: start low (256–512) for exploration; increase (2–8k) near convergence.  
- **Initial layout**: map logical qubits to strongest-fidelity hardware qubits; avoid SWAPs.  
- **Feature selection first**: 3–6 features often sweet-spot for today’s devices.

---

## 9) Worked hybrid pattern — “quantum head” on a small CNN

```python
# Pseudocode skeleton
# 1) CNN (tiny) -> features z (dim=4)
# 2) Encode z with 4-qubit feature map
# 3) PQC produces 2 expectations -> classical logistic layer

z = CNN_small(x_image)          # torch / tf module, outputs shape [B, 4]
phi = encode_angles(z)          # bind to ZZFeatureMap(4)
q_out = qnn(phi)                # expectations [B, 2]
logits = Linear(2, 1)(q_out)    # classical head
loss = BCEWithLogits(logits, y) # train end-to-end (PyTorch) or two-phase (TF)
```

---

## 10) Mini-exercises (answers in Appendix)

1. **Batching:** Given 200 samples, 40 parameters, and parameter-shift gradients, estimate the number of circuit evaluations per epoch with and without mini-batching (batch size 20).  
2. **Encoding choice:** You have 32 features but only 5 reliable qubits. Propose a pipeline using classical PCA + quantum re-uploading.  
3. **Noise budget:** With 1% two-qubit error and depth ~40 (20 two-qubit gates), estimate the multiplicative fidelity shrink on expectations and suggest two mitigations.  
4. **Torch vs TF:** Outline a two-phase training plan where the quantum head is trained with SPSA (PyTorch), then frozen and exported to a Keras model.  
5. **Kernel fallback:** Your PQC gradients collapse (barren plateau). Show how to swap the PQC head for a quantum kernel SVM using the same feature map.

---

## 11) Summary (Session 13)
- Hybrid = **quantum feature extractor** + **classical learner** or vice-versa.  
- Interfaces: encode data → PQC → decode expectations/probabilities → classical loss.  
- Train with parameter-shift/SPSA; batch to amortize latency; calibrate readout.  
- Qiskit ML integrates cleanly with **PyTorch** via `EstimatorQNN` + `TorchConnector`; TF requires a custom bridge or two-phase training.  
- Feature selection and topology-aware circuits are practical prerequisites for good results on NISQ.

---

## 12) Looking ahead
- **Next Session:** Quantum Convolutional Neural Networks (QCNNs) — translationally-equivariant circuits and pooling.  
- **Homework 4 (hybrid):**  
  1) Build a PyTorch hybrid (moons or breast-cancer) with `EstimatorQNN`; report accuracy vs. shots (256→4096).  
  2) Add readout calibration and show effect on BCE & accuracy.  
  3) (Bonus) Two-phase Keras integration using the TF wrapper above.

---

## Appendix — mini-exercise solutions (sketch)

1. **Evaluations/epoch:** parameter-shift → 2 evals/param.  
   - Full batch: $200 \times 2 \times 40 = 16{,}000$ circuit evals.  
   - Mini-batch (20): $10 \text{ batches} \times 2 \times 40 = 800$ evals per epoch (each eval processes 20 samples batched).  
2. **32→5 qubits:** PCA to 4 dims; encode with 4-qubit ZZ map; add **data re-uploading** (2–3 repeats) to boost expressivity.  
3. **Shrink:** roughly $(1-0.01)^{20}\approx 0.82$. Mitigate via (i) routing to best qubits + transpiler level 2–3, (ii) ZNE and dynamical decoupling.  
4. **Two-phase:** Train QNN head in PyTorch (SPSA) until convergence; export learned $\theta^*$. In Keras, wrap a `QNNLayer` that uses fixed $\theta^*$ (no gradients) and train surrounding classical layers.  
5. **Kernel swap:** Keep the same feature map $U_\phi(x)$; compute $K_{ij}=|\langle 0|U_\phi^\dagger(x_i)U_\phi(x_j)|0\rangle|^2$; train `SVC(kernel='precomputed')`. This removes variational parameters (no barren plateau) at the cost of $O(N^2)$ kernel evaluations.
